# XE5 LARGE core-matrix AMMA GEMM

A fully worked, cell-by-cell functional port of the sycl-tla example [`examples/cute/tutorial/xe5/xe5_adma_amma_large.cpp`](../../examples/cute/tutorial/xe5/xe5_adma_amma_large.cpp ), runnable on the CPU.

LARGE SLM core matrices (8192-byte descriptor tiles).

Every cell performs one operation on small concrete data and shows the matching Xe layout. Run them top to bottom. (A runnable script version lives in `xe5_adma_amma_large.py`.)

## Setup — data + a tiny layout printer

In [1]:
import numpy as np
np.set_printoptions(precision=2, suppress=True, linewidth=120)
from tensor_layouts import Layout, size
from tensor_layouts.analysis import is_bijective
from tensor_layouts.atoms_xe_common import make_slm_layout_elem, sizeof_bits

def show_layout(layout, n_rows, n_cols, rl="m", cl="k", max_r=8, max_c=8):
    "Print coord -> memory offset for a rank-2 layout (truncated)."
    R, C = min(n_rows, max_r), min(n_cols, max_c)
    print("      " + "".join((cl + str(j)).ljust(5) for j in range(C)) + (" ..." if C < n_cols else ""))
    for i in range(R):
        print((" " + rl + str(i)).ljust(6) + "".join(str(layout(i, j)).ljust(5) for j in range(C))
              + (" ..." if C < n_cols else ""))
    if R < n_rows:
        print("  ...  (%dx%d total)" % (n_rows, n_cols))

M, N, K = 32, 32, 16          # A rows, B rows, contraction
rng = np.random.default_rng(0)
A = rng.integers(-2, 3, size=(M, K)).astype(np.float32)   # A  (M x K)
B = rng.integers(-2, 3, size=(N, K)).astype(np.float32)   # B  (N x K), used as B^T
print("A", A.shape, " B", B.shape)
print("A[:4]:\n", A[:4])

A (32, 16)  B (32, 16)
A[:4]:
 [[ 2.  1.  0. -1. -1. -2. -2. -2. -2.  2.  1.  2.  0.  1.  2.  1.]
 [ 1.  0.  0.  2. -1.  2.  1. -2. -1.  2.  0. -2.  1.  1.  2. -2.]
 [-2.  2. -2.  0. -2. -1.  0.  0.  0. -2. -2. -2. -2.  1.  0.  1.]
 [-1.  1.  1. -1.  0.  2.  2.  2. -1.  1.  2.  1.  2.  1.  1. -1.]]


## Step — LARGE SLM load

In [2]:
# Xe5 LARGE core matrix: the SLM descriptor tile is larger (8192 B / 128 cols
# vs the 1024 B normal tile). The element layout is still a bijection.
slm_large = make_slm_layout_elem(sizeof_bits("bf16"), M, K)
print("LARGE SLM tile:", slm_large, " size:", size(slm_large), " bijective:", is_bijective(slm_large))
buf = np.zeros(size(slm_large), dtype=np.float32)
for m in range(M):
    for k in range(K):
        buf[slm_large(m, k)] = A[m, k]
A_back = np.array([[buf[slm_large(m, k)] for k in range(K)] for m in range(M)])
print("round-trip:", np.array_equal(A_back, A))

LARGE SLM tile: ((2, 4, 4, 1), (16, 1)) : ((16, 128, 32, 512), (1, 512))  size: 512  bijective: True
round-trip: True


## Step — MMA: S = A · Bᵀ

In [3]:
# MMA: the systolic multiply computes S = A . B^T.
S = A @ B.T
print("S = A . B^T  shape", S.shape, "\n", S[:4, :8], "...")
aM, aN, aK = 32, 32, 16
print("accumulator grid: %d x %d x %d atom tile(s)  (atom %dx%dx%d)"
      % (M // aM, N // aN, max(K // aK, 1), aM, aN, aK))
C = Layout((N, M), (M, 1))   # (thread=col of B, value=row of A) -> element, col-major M x N
print("C accumulator (thread, value) -> element offset:")
show_layout(C, N, M, rl="t", cl="v")

S = A . B^T  shape (32, 32) 
 [[ -5.  -6.   2.   3.   1.  -3.  -7.   9.]
 [ -1.   1.   4.   4.   0.   0.   5.  10.]
 [ -5.   7.   8.  -8.   1. -12.  -7.  -6.]
 [ 15. -14.   1.   0.  -1.  -4.  -2.   0.]] ...
accumulator grid: 1 x 1 x 1 atom tile(s)  (atom 32x32x16)
C accumulator (thread, value) -> element offset:
      v0   v1   v2   v3   v4   v5   v6   v7    ...
 t0   0    1    2    3    4    5    6    7     ...
 t1   32   33   34   35   36   37   38   39    ...
 t2   64   65   66   67   68   69   70   71    ...
 t3   96   97   98   99   100  101  102  103   ...
 t4   128  129  130  131  132  133  134  135   ...
 t5   160  161  162  163  164  165  166  167   ...
 t6   192  193  194  195  196  197  198  199   ...
 t7   224  225  226  227  228  229  230  231   ...
  ...  (32x32 total)


## Step — ADMA store: SLM → gmem

In [4]:
# ADMA store: write S back to global memory through the SLM core matrix
# (reverse of the load) and confirm it is loss-less.
res = S
slm_o = make_slm_layout_elem(sizeof_bits("bf16"), res.shape[0], res.shape[1])
print("result corner (to be stored):\n", res[:3, :6], "...")
obuf = np.zeros(size(slm_o), dtype=res.dtype)
for i in range(res.shape[0]):
    for j in range(res.shape[1]):
        obuf[slm_o(i, j)] = res[i, j]
print("SLM store offsets for rows 0..3, col 0:", [slm_o(i, 0) for i in range(min(4, res.shape[0]))])
back = np.array([[obuf[slm_o(i, j)] for j in range(res.shape[1])] for i in range(res.shape[0])])
print("store round-trip matches result:", np.array_equal(back, res))

result corner (to be stored):
 [[ -5.  -6.   2.   3.   1.  -3.]
 [ -1.   1.   4.   4.   0.   0.]
 [ -5.   7.   8.  -8.   1. -12.]] ...
SLM store offsets for rows 0..3, col 0: [0, 16, 128, 144]
store round-trip matches result: True


## Recap

large SLM load → MMA → store.